# Extracción de Datos de Videoanálisis (Longomatch)

## Objetivo

Construir un pipeline automatizado que extraiga eventos de partidos (goles, remates y pases) desde archivos Excel exportados por Longomatch, los limpie, los cruce con las tablas dimensionales del proyecto (BeSoccer) y genere una tabla `fact_eventos_videoanalisis` lista para Power BI.

## Contexto del proyecto

- **Competición**: 2ª RFEF Grupo 3, temporada 25-26
- **Fuente de datos**: Archivos `.xlsx` exportados desde Longomatch tras el etiquetado de vídeo de cada partido
- **Tablas dimensionales** (scraping BeSoccer): `dim_equipos`, `dim_partidos`, `dim_jugadores`, `fact_appearances`
- **Destino final**: Power BI, donde se cruzarán datos de scraping con datos de videoanálisis

## Estructura del notebook

1. **Configuración** — imports, rutas, carga de dimensionales
2. **EDA estructural** — análisis de la estructura de los archivos Excel
3. **Detalle por bloque** — columnas, tipos, valores por tipo de evento
4. **Validación cruzada** — matching con tablas dimensionales
5. **Funciones de limpieza** — transformación de campos
6. **Pipeline de procesamiento** — extracción completa por archivo
7. **Modelo final** — esquema `fact_eventos` para Power BI
8. **Ejecución y exportación** — pipeline de 1 click


In [ ]:
# --- Imports y configuración ---

from pathlib import Path
import pandas as pd
import openpyxl
import unicodedata
import datetime
import logging
import warnings
warnings.filterwarnings('ignore')

# Rutas del proyecto
PROJECT_ROOT = Path(r"C:\Users\manue\OneDrive\Sports Data Campus\Máster Python\Proyecto Sant Andreu")
DATA_PROCESSED = PROJECT_ROOT / "data_processed"
DATA_PARTIDOS = PROJECT_ROOT / "partidos"

# Archivos disponibles
ARCHIVOS_EXCEL = sorted(DATA_PARTIDOS.glob("*.xlsx"))

print(f"Archivos en {DATA_PARTIDOS.name}/:")
for f in ARCHIVOS_EXCEL:
    print(f"  [OK] {f.name}")

# Tablas dimensionales
dim_equipos = pd.read_parquet(DATA_PROCESSED / "dim_equipos.parquet")
dim_partidos = pd.read_parquet(DATA_PROCESSED / "dim_partidos.parquet")
dim_jugadores = pd.read_parquet(DATA_PROCESSED / "dim_jugadores.parquet")
fact_appearances = pd.read_parquet(DATA_PROCESSED / "fact_appearances.parquet")

print(f"\ndim_equipos: {dim_equipos.shape} | dim_partidos: {dim_partidos.shape}")
print(f"dim_jugadores: {dim_jugadores.shape} | fact_appearances: {fact_appearances.shape}")


## 1. EDA Estructural

Los archivos Excel de Longomatch comparten una estructura consistente en la hoja `Estadísticas del proyecto`:

- **Filas 1-8**: Metadatos del partido (proyecto, fecha, equipos, temporada)
- **Bloques de eventos**: Apilados verticalmente, cada uno con una fila de título, una de cabecera y N filas de datos

Los bloques siguen siempre el mismo **orden**: GOL Local, REMATES Visitante, GOL Visitante, REMATES Local, PASE Local, PASE Visitante, PERIODO, Sustituciones.

**Hallazgo clave**: El número de filas por bloque **no es fijo**. Longomatch reserva un mínimo de ~14 filas pero lo amplía dinámicamente si hay más eventos. Por ello, el parser detecta los bloques por sus **títulos ancla** en vez de usar rangos hardcodeados.


In [ ]:
# --- Parser dinámico: detección de bloques por títulos ancla ---

# Metadatos: siempre en filas fijas 1-8
METADATOS_FILAS = {
    'proyecto': 2, 'fecha': 3, 'competicion': 4, 'temporada': 5,
    'equipo_local': 6, 'equipo_visitante': 7, 'resultado': 8,
}

# Títulos ancla que marcan el inicio de cada bloque (col A, fila de título)
TITULOS_BLOQUE = {
    'GOL Local':         {'tipo': 'GOL',     'equipo_lado': 'Local'},
    'REMATES Visitante': {'tipo': 'REMATE',  'equipo_lado': 'Visitante'},
    'GOL Visitante':     {'tipo': 'GOL',     'equipo_lado': 'Visitante'},
    'REMATES Local':     {'tipo': 'REMATE',  'equipo_lado': 'Local'},
    'PASE Local':        {'tipo': 'PASE',    'equipo_lado': 'Local'},
    'PASE Visitante':    {'tipo': 'PASE',    'equipo_lado': 'Visitante'},
    'PERIODO':           {'tipo': 'PERIODO', 'equipo_lado': None},
    'Sustituciones':     {'tipo': 'SUSTITUCION', 'equipo_lado': None},
}

# Esquemas de columnas por tipo de evento (hoja principal, max 15 cols)
COLUMNAS_GOL = [
    'Evento', 'Tiempo', 'Inicio', 'Fin', 'Equipo', 'Jugadores',
    'Situacion', 'FieldXfrom', 'FieldYfrom', 'FieldXto', 'FieldYto',
    'GoalX', 'GoalY', '_v1', '_v2'
]
COLUMNAS_REMATE = [
    'Evento', 'Tiempo', 'Inicio', 'Fin', 'Equipo', 'Jugadores',
    'Situacion', 'Resultado', 'FieldXfrom', 'FieldYfrom', 'FieldXto', 'FieldYto',
    'GoalX', 'GoalY', '_v1'
]
COLUMNAS_PASE = [
    'Evento', 'Tiempo', 'Inicio', 'Fin', 'Equipo', 'Jugadores',
    'Forma', 'Sector', 'FieldXfrom', 'FieldYfrom', 'FieldXto', 'FieldYto',
    '_v1', '_v2', '_v3'
]
COLUMNAS_PERIODO = [
    'Evento', 'Tiempo', 'Inicio', 'Fin', 'Equipo', 'Jugadores',
    'Periodo', None, None, None, None, None, None, None, None
]

ESQUEMA_POR_TIPO = {
    'GOL': COLUMNAS_GOL, 'REMATE': COLUMNAS_REMATE,
    'PASE': COLUMNAS_PASE, 'PERIODO': COLUMNAS_PERIODO,
}


def leer_archivo_raw(filepath):
    """
    Lee un xlsx de Nacsport con detección DINÁMICA de bloques.
    Busca los títulos ancla en la columna A para delimitar cada bloque,
    en vez de asumir rangos fijos de filas.
    
    Returns:
        metadatos: dict con info del partido
        bloques_raw: dict {nombre_bloque: {header, filas, n_datos, tipo, equipo_lado}}
    """
    wb = openpyxl.load_workbook(filepath, read_only=True)
    ws = wb[wb.sheetnames[0]]
    
    # Cargar todas las filas en memoria
    todas_filas = {}
    for i, row in enumerate(ws.iter_rows(values_only=True), 1):
        todas_filas[i] = list(row)
    wb.close()
    
    max_fila = max(todas_filas.keys())
    
    # --- Metadatos (filas fijas 1-8) ---
    metadatos = {}
    for campo, fila_num in METADATOS_FILAS.items():
        row = todas_filas.get(fila_num, [None, None])
        metadatos[campo] = row[1] if len(row) > 1 else None
    
    # --- Detectar bloques dinámicamente ---
    # 1. Encontrar las filas donde aparece cada título ancla
    posiciones_titulo = []
    for fila_num in sorted(todas_filas.keys()):
        row = todas_filas[fila_num]
        if row and row[0] and isinstance(row[0], str):
            valor = row[0].strip()
            if valor in TITULOS_BLOQUE:
                config = TITULOS_BLOQUE[valor]
                posiciones_titulo.append({
                    'titulo': valor,
                    'fila_titulo': fila_num,
                    'tipo': config['tipo'],
                    'equipo_lado': config['equipo_lado'],
                })
    
    # 2. Para cada bloque: header = título+1, datos = título+2 hasta siguiente título-1
    bloques_raw = {}
    for idx, bloque_info in enumerate(posiciones_titulo):
        fila_titulo = bloque_info['fila_titulo']
        fila_header = fila_titulo + 1
        fila_datos_inicio = fila_titulo + 2
        
        # El final de datos es la fila anterior al siguiente título (o fin de archivo)
        if idx + 1 < len(posiciones_titulo):
            fila_datos_fin = posiciones_titulo[idx + 1]['fila_titulo'] - 1
        else:
            fila_datos_fin = max_fila
        
        # Header
        header_row = todas_filas.get(fila_header, [])
        header = [v for v in header_row if v is not None]
        
        # Filas de datos (filtrar vacías)
        filas_datos = []
        for f in range(fila_datos_inicio, fila_datos_fin + 1):
            row = todas_filas.get(f, [])
            if row and row[0] is not None and row[0] != '':
                filas_datos.append(row)
        
        # Nombre del bloque: TIPO_LADO (ej: GOL_LOCAL)
        tipo = bloque_info['tipo']
        lado = bloque_info['equipo_lado']
        nombre = f"{tipo}_{lado.upper()}" if lado else tipo
        
        bloques_raw[nombre] = {
            'header': header,
            'filas': filas_datos,
            'n_datos': len(filas_datos),
            'tipo': tipo,
            'equipo_lado': lado,
            'fila_titulo': fila_titulo,
            'rango': (fila_datos_inicio, fila_datos_fin),
        }
    
    return metadatos, bloques_raw


# === EDA: ejecutar sobre todos los archivos ===
for archivo in ARCHIVOS_EXCEL:
    print(f"{'='*70}")
    print(f"ARCHIVO: {archivo.name}")
    print(f"{'='*70}")
    
    meta, bloques = leer_archivo_raw(archivo)
    
    print(f"\n--- METADATOS ---")
    for k, v in meta.items():
        print(f"  {k:<20}: {v}")
    
    print(f"\n--- BLOQUES (detección dinámica) ---")
    print(f"  {'Bloque':<22} {'Eventos':>7}  {'Rango filas':<15} Header")
    print(f"  {'-'*80}")
    for nombre, info in bloques.items():
        if info['tipo'] == 'SUSTITUCION':
            continue  # no nos interesa por ahora
        rango = f"{info['rango'][0]}-{info['rango'][1]}"
        print(f"  {nombre:<22} {info['n_datos']:>7}  {rango:<15} {info['header'][:6]}...")
    
    goles_l = bloques.get('GOL_LOCAL', {}).get('n_datos', 0)
    goles_v = bloques.get('GOL_VISITANTE', {}).get('n_datos', 0)
    total = sum(b['n_datos'] for b in bloques.values() if b['tipo'] not in ('PERIODO', 'SUSTITUCION'))
    print(f"\n  Resultado calculado: {goles_l} - {goles_v} | Total eventos: {total}")
    print()


## 2. Detalle por bloque — columnas, tipos y valores

Cada tipo de evento tiene un esquema de columnas diferente:

| Tipo | Columnas específicas | Coordenadas |
|---|---|---|
| **GOL** | `Situacion` | Campo (from/to) + Portería (GoalX/Y) |
| **REMATE** | `Situacion`, `Resultado` | Campo (from/to) + Portería (GoalX/Y, nulo si desviado) |
| **PASE** | `Forma`, `Sector` | Campo (from/to), sin portería |

**Sistema de coordenadas (convención de etiquetado):**
- **Campo**: coordenadas normalizadas 0-1 en un campo horizontal. `(0,0)` = esquina inferior izquierda. Todos los ataques se registran hacia la derecha (X=1 = portería rival)
- **Portería**: vista frontal. `(0,0)` = base del palo derecho del portero. `(1,0)` = base del palo izquierdo


In [ ]:
# --- Detalle: columnas, tipos y valores por bloque ---

for archivo in ARCHIVOS_EXCEL[:1]:  # solo el primer archivo como ejemplo
    print(f"ARCHIVO: {archivo.name}\n")
    meta, bloques = leer_archivo_raw(archivo)
    
    for nombre_bloque, info in bloques.items():
        if info['n_datos'] == 0 or info['tipo'] == 'SUSTITUCION':
            continue
        
        tipo = info['tipo']
        esquema = ESQUEMA_POR_TIPO.get(tipo)
        if not esquema:
            continue
        
        print(f"  [{nombre_bloque}] — {info['n_datos']} eventos")
        
        filas_norm = []
        for fila in info['filas']:
            fila_ext = (fila + [None] * 15)[:15]
            filas_norm.append(fila_ext)
        
        df_b = pd.DataFrame(filas_norm, columns=esquema)
        df_b = df_b[[c for c in df_b.columns if c and not c.startswith('_')]]
        
        print(f"  {'Columna':<15} {'Tipo':<12} {'Nulos':>5}  Valores únicos")
        print(f"  {'-'*65}")
        for col in df_b.columns:
            s = df_b[col]
            nulos = s.isna().sum() + (s == '').sum()
            tipo_c = type(s.dropna().iloc[0]).__name__ if s.dropna().shape[0] > 0 else 'nulo'
            unicos = s[s.notna() & (s != '')].unique()
            muestra = list(unicos[:4])
            if len(unicos) > 4:
                muestra.append(f'...+{len(unicos)-4}')
            print(f"  {col:<15} {tipo_c:<12} {nulos:>5}  {muestra}")
        print()

In [ ]:
# --- Funciones de matching con tablas dimensionales ---

def normalizar_nombre(nombre):
    """Quitar tildes, upper, strip."""
    nfkd = unicodedata.normalize('NFKD', str(nombre))
    return ''.join(c for c in nfkd if not unicodedata.combining(c)).upper().strip()


def buscar_team_id(nombre_excel, dim_equipos):
    """Busca team_id: primero exacto (normalizado), luego por contención."""
    nombre_norm = normalizar_nombre(nombre_excel)
    for _, row in dim_equipos.iterrows():
        if normalizar_nombre(row['nombre_equipo']) == nombre_norm:
            return row['team_id'], row['nombre_equipo'], 'exacto'
    for _, row in dim_equipos.iterrows():
        dim_norm = normalizar_nombre(row['nombre_equipo'])
        if dim_norm in nombre_norm or nombre_norm in dim_norm:
            return row['team_id'], row['nombre_equipo'], 'contención'
    return None, None, 'SIN MATCH'


def buscar_match_id(home_team_id, away_team_id, goles_local, goles_visitante, dim_partidos):
    """Busca match_id cruzando equipos + resultado."""
    candidatos = dim_partidos[
        (dim_partidos['home_team_id'] == home_team_id) &
        (dim_partidos['away_team_id'] == away_team_id)
    ]
    if candidatos.empty:
        return None, "No se encontró partido con esos equipos"
    con_resultado = candidatos[
        (candidatos['score_home'] == goles_local) &
        (candidatos['score_away'] == goles_visitante)
    ]
    if len(con_resultado) == 1:
        row = con_resultado.iloc[0]
        return row['match_id'], f"Match único: J{row['jornada']} {row['fecha']}"
    elif len(con_resultado) > 1:
        return None, f"AMBIGUO: {len(con_resultado)} partidos con mismo resultado"
    else:
        info = candidatos[['match_id', 'fecha', 'score_home', 'score_away']].to_string()
        return None, f"Resultado no coincide.\nCandidatos:\n{info}"


# === Validación cruzada ===
print("VALIDACIÓN CRUZADA CON TABLAS DIMENSIONALES\n")

for archivo in ARCHIVOS_EXCEL:
    meta, bloques = leer_archivo_raw(archivo)
    home_id, home_name, hm = buscar_team_id(meta['equipo_local'], dim_equipos)
    away_id, away_name, am = buscar_team_id(meta['equipo_visitante'], dim_equipos)
    
    goles_l = bloques.get('GOL_LOCAL', {}).get('n_datos', 0)
    goles_v = bloques.get('GOL_VISITANTE', {}).get('n_datos', 0)
    
    match_id, match_info = buscar_match_id(home_id, away_id, goles_l, goles_v, dim_partidos)
    
    print(f"--- {meta['equipo_local']} vs {meta['equipo_visitante']} ---")
    print(f"  Equipos: {home_name} (id={home_id}, {hm}) vs {away_name} (id={away_id}, {am})")
    print(f"  Resultado: {goles_l}-{goles_v} | match_id={match_id} — {match_info}")
    
    # Validar dorsales
    if match_id:
        apps = fact_appearances[fact_appearances['match_id'] == match_id]
        dorsales = set()
        for info in bloques.values():
            if info['tipo'] in ('PERIODO', 'SUSTITUCION') or info['n_datos'] == 0:
                continue
            esquema = ESQUEMA_POR_TIPO.get(info['tipo'], [])
            if 'Jugadores' not in esquema:
                continue
            idx_j = esquema.index('Jugadores')
            for fila in info['filas']:
                fila_ext = (fila + [None] * 15)[:15]
                jug = fila_ext[idx_j]
                if jug and str(jug).strip():
                    try:
                        dorsales.add((int(str(jug).strip().split('-')[0]), info['equipo_lado']))
                    except ValueError:
                        pass
        
        resueltos = 0
        for dorsal, lado in sorted(dorsales):
            side = 'local' if lado == 'Local' else 'visitor'
            m = apps[(apps['shirt_number'] == dorsal) & (apps['side'] == side)]
            if not m.empty:
                resueltos += 1
                print(f"    Dorsal {dorsal:>2} ({lado:<10}) -> {m.iloc[0]['player_name']}")
            else:
                print(f"    Dorsal {dorsal:>2} ({lado:<10}) -> SIN MATCH")
        print(f"  Dorsales: {resueltos}/{len(dorsales)} resueltos\n")


## 4. Funciones de limpieza y transformación

Transformaciones aplicadas a cada evento:

- **Tiempos**: `datetime.time` → `timedelta` (permite aritmética)
- **Minuto de juego**: calculado restando el inicio del periodo correspondiente al timestamp de video. PT = 0-45, ST = 45+
- **Dorsales**: `"7-7 "` → `7` (int)
- **Campos categóricos**: se mantienen tal cual (`Situacion`, `Resultado`, `Forma`, `Sector`)
- **Coordenadas de portería**: nulas para remates desviados (nunca impactaron en el marco)


In [ ]:
# --- Funciones de limpieza ---

def time_to_timedelta(t):
    """datetime.time -> timedelta."""
    if t is None or not isinstance(t, datetime.time):
        return pd.NaT
    return pd.Timedelta(hours=t.hour, minutes=t.minute, seconds=t.second, 
                        microseconds=t.microsecond)


def extraer_dorsal(jugadores_raw):
    """'7-7 ' -> 7."""
    if jugadores_raw is None or str(jugadores_raw).strip() == '':
        return None
    try:
        return int(str(jugadores_raw).strip().split('-')[0])
    except (ValueError, IndexError):
        return None


def extraer_periodos(bloques_raw):
    """Extrae timestamps de los 4 marcadores de periodo."""
    periodos = {}
    mapping = {
        'Inicio PT': 'inicio_pt', 'Final PT': 'final_pt',
        'Incio ST': 'inicio_st', 'Inicio ST': 'inicio_st',  # typo Nacsport
        'Final ST': 'final_st',
    }
    for fila in bloques_raw['PERIODO']['filas']:
        fila_ext = (fila + [None] * 15)[:15]
        periodo_nombre = fila_ext[6]
        tiempo = fila_ext[1]
        if periodo_nombre in mapping:
            periodos[mapping[periodo_nombre]] = time_to_timedelta(tiempo)
    return periodos


def calcular_minuto_juego(tiempo_video_td, periodos):
    """Timestamp de video -> minuto de juego real (PT: 0-45, ST: 45+)."""
    if pd.isna(tiempo_video_td):
        return None
    inicio_pt = periodos.get('inicio_pt')
    inicio_st = periodos.get('inicio_st')
    if pd.isna(inicio_pt) or pd.isna(inicio_st):
        return None
    if tiempo_video_td < inicio_st:
        return round((tiempo_video_td - inicio_pt).total_seconds() / 60.0, 2)
    else:
        return round(45.0 + (tiempo_video_td - inicio_st).total_seconds() / 60.0, 2)


def procesar_bloque(nombre_bloque, info, periodos):
    """Convierte un bloque raw en lista de dicts con esquema unificado."""
    if info['n_datos'] == 0:
        return []
    
    tipo = info['tipo']
    equipo_lado = info['equipo_lado']
    esquema = ESQUEMA_POR_TIPO.get(tipo)
    if not esquema:
        return []
    
    filas_limpias = []
    for fila in info['filas']:
        fila_ext = (fila + [None] * 15)[:15]
        raw = {col: fila_ext[i] for i, col in enumerate(esquema) if col and not col.startswith('_')}
        
        evento = {
            'tipo_evento': tipo,
            'equipo_lado': equipo_lado,
            'evento_raw': raw.get('Evento'),
            'tiempo_video': time_to_timedelta(raw.get('Tiempo')),
            'inicio_video': time_to_timedelta(raw.get('Inicio')),
            'fin_video': time_to_timedelta(raw.get('Fin')),
            'dorsal': extraer_dorsal(raw.get('Jugadores')),
            'situacion': raw.get('Situacion'),
            'resultado_remate': raw.get('Resultado') if tipo == 'REMATE' else None,
            'forma_pase': raw.get('Forma') if tipo == 'PASE' else None,
            'sector_pase': raw.get('Sector') if tipo == 'PASE' else None,
            'field_x_from': raw.get('FieldXfrom'),
            'field_y_from': raw.get('FieldYfrom'),
            'field_x_to': raw.get('FieldXto'),
            'field_y_to': raw.get('FieldYto'),
            'goal_x': raw.get('GoalX') if tipo in ('GOL', 'REMATE') else None,
            'goal_y': raw.get('GoalY') if tipo in ('GOL', 'REMATE') else None,
        }
        evento['minuto_juego'] = calcular_minuto_juego(evento['tiempo_video'], periodos)
        evento['periodo'] = ('PT' if evento['minuto_juego'] < 45 else 'ST') if evento['minuto_juego'] is not None else None
        
        filas_limpias.append(evento)
    return filas_limpias


print("Funciones de limpieza definidas.")


## 5. Pipeline de procesamiento

La función `procesar_archivo_completo` encapsula el flujo completo para un archivo:

1. Leer Excel con parser dinámico
2. Resolver `team_id` para local y visitante
3. Calcular resultado desde eventos GOL
4. Buscar `match_id` en `dim_partidos`
5. Extraer periodos de referencia temporal
6. Procesar cada bloque de eventos
7. Resolver dorsales → jugadores vía `fact_appearances`
8. Ordenar por tiempo de vídeo


In [ ]:
# --- Pipeline de procesamiento por archivo ---

def procesar_archivo_completo(filepath, dim_equipos, dim_partidos, fact_appearances):
    """Pipeline completo: Excel -> DataFrame limpio con contexto del partido."""
    meta, bloques = leer_archivo_raw(filepath)
    
    # Resolver equipos
    home_id, home_name, _ = buscar_team_id(meta['equipo_local'], dim_equipos)
    away_id, away_name, _ = buscar_team_id(meta['equipo_visitante'], dim_equipos)
    if not home_id or not away_id:
        raise ValueError(f"No se pudo resolver equipos: {meta['equipo_local']} / {meta['equipo_visitante']}")
    
    # Resultado calculado
    goles_local = bloques.get('GOL_LOCAL', {}).get('n_datos', 0)
    goles_visitante = bloques.get('GOL_VISITANTE', {}).get('n_datos', 0)
    
    # Buscar match_id
    match_id, match_info = buscar_match_id(home_id, away_id, goles_local, goles_visitante, dim_partidos)
    if not match_id:
        print(f"  [WARN] {filepath.name}: {match_info}")
    
    # Periodos
    periodos = extraer_periodos(bloques)
    
    # Procesar bloques
    todos_eventos = []
    for nombre, info in bloques.items():
        if info['tipo'] in ('PERIODO', 'SUSTITUCION'):
            continue
        todos_eventos.extend(procesar_bloque(nombre, info, periodos))
    
    if not todos_eventos:
        return pd.DataFrame()
    
    df = pd.DataFrame(todos_eventos)
    
    # Contexto del partido
    df['match_id'] = match_id
    df['home_team_id'] = home_id
    df['home_team_name'] = home_name
    df['away_team_id'] = away_id
    df['away_team_name'] = away_name
    df['goles_local'] = goles_local
    df['goles_visitante'] = goles_visitante
    df['temporada'] = meta.get('temporada')
    
    # Fecha y jornada desde dim_partidos
    if match_id:
        p = dim_partidos[dim_partidos['match_id'] == match_id]
        df['fecha_partido'] = p.iloc[0]['fecha'] if not p.empty else None
        df['jornada'] = p.iloc[0]['jornada'] if not p.empty else None
    else:
        df['fecha_partido'] = None
        df['jornada'] = None
    
    # Resolver dorsales -> jugadores vía fact_appearances
    # NOTA: player_sk es la clave sustituta coordinada con dim_jugadores.
    # player_id puede ser NULL para jugadores sin ficha BeSoccer, pero player_sk NUNCA es NULL.
    df['player_sk'] = None
    df['player_id'] = None
    df['player_name'] = None
    if match_id:
        apps = fact_appearances[fact_appearances['match_id'] == match_id]
        for idx, row in df.iterrows():
            if pd.isna(row['dorsal']):
                continue
            side = 'local' if row['equipo_lado'] == 'Local' else 'visitor'
            m = apps[(apps['shirt_number'] == row['dorsal']) & (apps['side'] == side)]
            if not m.empty:
                df.at[idx, 'player_sk'] = m.iloc[0]['player_sk']
                df.at[idx, 'player_id'] = m.iloc[0]['player_id']
                df.at[idx, 'player_name'] = m.iloc[0]['player_name']
    
    # Ordenar y limpiar tipos
    df = df.sort_values('tiempo_video').reset_index(drop=True)
    for col in ['field_x_to', 'field_x_from', 'field_y_from', 'field_y_to']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    return df


# === Procesar todos los archivos ===
dataframes = []
for archivo in ARCHIVOS_EXCEL:
    print(f"Procesando: {archivo.name}")
    df = procesar_archivo_completo(archivo, dim_equipos, dim_partidos, fact_appearances)
    dataframes.append(df)
    print(f"  -> {len(df)} eventos | Jugadores sin resolver: {df['player_name'].isna().sum()}")

df_eventos = pd.concat(dataframes, ignore_index=True)
print(f"\nTotal: {len(df_eventos)} eventos de {df_eventos['match_id'].nunique()} partidos")


## 6. Modelo final — `fact_eventos_videoanalisis`

Esquema de la tabla de hechos con 35 columnas, diseñado para cruzarse en Power BI:

| Relación | Columna fact_eventos | Tabla dimensional |
|---|---|---|
| Partido | `match_id` | `dim_partidos.match_id` |
| Equipo ejecutor | `team_id` | `dim_equipos.team_id` |
| Jugador (clave PK) | `player_sk` | `dim_jugadores.player_sk` ← **usar esta** |
| Jugador (BeSoccer) | `player_id` | `dim_jugadores.player_id` (puede ser NULL) |
| Equipo local | `home_team_id` | `dim_equipos.team_id` |
| Equipo visitante | `away_team_id` | `dim_equipos.team_id` |

**Nota sobre `player_sk` vs `player_id`:**
- `player_sk` es la clave sustituta (surrogate key) de `dim_jugadores`. Siempre tiene valor para jugadores identificados por dorsal.
- `player_id` es el ID de BeSoccer. Es NULL para jugadores que no tienen ficha oficial en BeSoccer (17 jugadores en la dimensión).
- La relación en Power BI debe hacerse sobre `player_sk`, no sobre `player_id`.
- Los únicos eventos sin `player_sk` son los etiquetados con dorsal 99 (código "no sé quién es") o sin dorsal.

In [ ]:
# --- Modelo final y exportación ---

def construir_fact_eventos(df_raw):
    """Construye la tabla fact_eventos con esquema final para Power BI."""
    df = df_raw.copy()
    
    # Equipo que ejecuta la acción
    df['team_id'] = df.apply(
        lambda r: r['home_team_id'] if r['equipo_lado'] == 'Local' else r['away_team_id'], axis=1)
    df['team_name'] = df.apply(
        lambda r: r['home_team_name'] if r['equipo_lado'] == 'Local' else r['away_team_name'], axis=1)
    
    # Resultado como string
    df['resultado'] = df['goles_local'].astype(str) + '-' + df['goles_visitante'].astype(str)
    
    # Tiempos en segundos (Power BI no maneja timedelta)
    for col_td in ['tiempo_video', 'inicio_video', 'fin_video']:
        df[col_td.replace('video', 'seg')] = df[col_td].dt.total_seconds()
    
    # Minuto entero para filtros
    df['minuto_juego_int'] = df['minuto_juego'].apply(lambda x: int(x) if pd.notna(x) else None)
    
    # Selección y orden de columnas
    # player_sk: clave sustituta coordinada con dim_jugadores (NUNCA NULL si dorsal resuelto)
    # player_id: ID BeSoccer, puede ser NULL para jugadores sin ficha oficial
    columnas = [
        'match_id', 'fecha_partido', 'jornada', 'temporada',
        'home_team_id', 'home_team_name', 'away_team_id', 'away_team_name',
        'goles_local', 'goles_visitante', 'resultado',
        'tipo_evento', 'equipo_lado', 'team_id', 'team_name',
        'player_sk', 'player_id', 'player_name', 'dorsal',
        'periodo', 'minuto_juego', 'minuto_juego_int',
        'tiempo_seg', 'inicio_seg', 'fin_seg',
        'situacion', 'resultado_remate', 'forma_pase', 'sector_pase',
        'field_x_from', 'field_y_from', 'field_x_to', 'field_y_to',
        'goal_x', 'goal_y',
    ]
    df = df[columnas].copy()
    
    # Tipos correctos
    for col in ['match_id', 'jornada', 'home_team_id', 'away_team_id',
                'goles_local', 'goles_visitante', 'team_id', 'dorsal',
                'minuto_juego_int', 'player_sk', 'player_id']:
        df[col] = df[col].astype('Int64')
    
    return df


# === Construir y validar ===
fact_eventos = construir_fact_eventos(df_eventos)

print(f"fact_eventos: {fact_eventos.shape[0]} filas x {fact_eventos.shape[1]} columnas\n")

# Validación player_sk
print("Validación player_sk:")
print(f"  player_sk nulos: {fact_eventos['player_sk'].isnull().sum()} (solo dorsal 99 / sin dorsal)")
print(f"  player_id nulos: {fact_eventos['player_id'].isnull().sum()} (jugadores sin ficha BeSoccer)")
print(f"  player_name nulos: {fact_eventos['player_name'].isnull().sum()}")

# Distribución
print("\nDistribución por partido y tipo:")
pivot = fact_eventos.groupby(
    ['match_id', 'home_team_name', 'away_team_name', 'resultado', 'tipo_evento']
).size().unstack(fill_value=0)
print(pivot.to_string())

# Validación de resultados
print("\nValidación resultado calculado vs BeSoccer:")
for mid in fact_eventos['match_id'].unique():
    sub = fact_eventos[fact_eventos['match_id'] == mid].iloc[0]
    p = dim_partidos[dim_partidos['match_id'] == mid].iloc[0]
    calc = f"{sub['goles_local']}-{sub['goles_visitante']}"
    oficial = f"{p['score_home']}-{p['score_away']}"
    print(f"  [{'OK' if calc == oficial else 'MISMATCH'}] {sub['home_team_name']} vs {sub['away_team_name']}: {calc} (calc) vs {oficial} (oficial)")


## 7. Pipeline de ejecución (1 click)

Función `pipeline_videoanalisis()` que procesa toda la carpeta `partidos/` y exporta:
- `fact_eventos_videoanalisis.parquet` — para Power BI
- `fact_eventos_videoanalisis.csv` — para inspección rápida

**Uso**: añadir nuevos `.xlsx` a la carpeta y re-ejecutar.


In [ ]:
# --- Pipeline 1-click ---

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
logger = logging.getLogger('videoanalisis')


def pipeline_videoanalisis(carpeta_partidos=None, exportar=True):
    """
    Pipeline completo: carpeta de xlsx -> fact_eventos exportado.
    """
    if carpeta_partidos is None:
        carpeta_partidos = DATA_PARTIDOS
    carpeta_partidos = Path(carpeta_partidos)
    
    _dim_eq = pd.read_parquet(DATA_PROCESSED / "dim_equipos.parquet")
    _dim_par = pd.read_parquet(DATA_PROCESSED / "dim_partidos.parquet")
    _fact_app = pd.read_parquet(DATA_PROCESSED / "fact_appearances.parquet")
    
    archivos = sorted(carpeta_partidos.glob("*.xlsx"))
    if not archivos:
        logger.warning(f"No se encontraron archivos en {carpeta_partidos}")
        return pd.DataFrame()
    
    logger.info(f"Encontrados {len(archivos)} archivos")
    
    dataframes, errores = [], []
    for archivo in archivos:
        try:
            df = procesar_archivo_completo(archivo, _dim_eq, _dim_par, _fact_app)
            n_sin = df['player_name'].isna().sum()
            if n_sin > 0:
                logger.warning(f"  {archivo.name}: {n_sin} eventos sin jugador")
            dataframes.append(df)
            logger.info(f"  OK: {archivo.name} -> {len(df)} eventos")
        except Exception as e:
            logger.error(f"  ERROR: {archivo.name} -> {e}")
            errores.append((archivo.name, str(e)))
    
    if not dataframes:
        return pd.DataFrame()
    
    fact = construir_fact_eventos(pd.concat(dataframes, ignore_index=True))
    
    logger.info(f"\nArchivos: {len(dataframes)}/{len(archivos)} | Eventos: {len(fact)} | Partidos: {fact['match_id'].nunique()}")
    
    if exportar:
        fact.to_parquet(DATA_PROCESSED / "fact_eventos_videoanalisis.parquet", index=False)
        fact.to_csv(DATA_PROCESSED / "fact_eventos_videoanalisis.csv", index=False, sep=';', decimal=',')
        logger.info(f"Exportado en {DATA_PROCESSED.name}/")
    
    if errores:
        logger.error(f"Errores: {errores}")
    
    return fact


# === EJECUTAR ===
fact_final = pipeline_videoanalisis()
print(f"\n{fact_final.shape[0]} eventos listos para Power BI")
print(fact_final[['tipo_evento', 'equipo_lado', 'minuto_juego', 'player_name', 'situacion', 'forma_pase']].head(10).to_string())
